# Utilization cutoff sensitivity (inputs)

Baseline and memory-cutoff scenarios that `utilization_cutoff_uniform_pp.py` checks before computing the ±3 and ±5 percentage-point sensitivity in Section 4.4.

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()
from cache_paths import require_frozen_cache
ONLINE_CACHE = require_frozen_cache('online_ir2', ['container_events.csv'])

from cohort_contract import current_contract
LATEST_CONTRACT=current_contract()


## 1. Inputs and checks

In [ ]:
_corrected_contract=__import__('json').loads((ROOT/'inputs/preprocessing_validation.json').read_text())
# Inputs and outputs stay inside the repository.
from pathlib import Path
from IPython.display import display as _cleanup_display, HTML as _CleanupHTML

RESULTS = RESULTS_REVISION
RESULTS.mkdir(parents=True, exist_ok=True)

def display(*objects, **kwargs):
    """Display-only terminology; dataframe values and calculation keys are unchanged."""
    for obj in objects:
        html = obj.data if isinstance(obj, _CleanupHTML) else (
            obj._repr_html_() if hasattr(obj, "_repr_html_") else None
        )
        if isinstance(html, str):
            html = html.replace("GPU-Enabled", "Integer-GPU")
            _cleanup_display(_CleanupHTML(html), **kwargs)
        else:
            _cleanup_display(obj, **kwargs)

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display as _cleanup_display

DATA_ROOT = Path(str(PROCESSED_DATA))
USAGE_PATH = DATA_ROOT / "UsageLog20230401_20251130_20260301_20260831.csv"
TRACE_PATH = DATA_ROOT / "TimeSeries_20240926_20251130_20260301_20260831.parquet"
ONLINE_DIR = Path(str(REVISION_CODE / 'online_ir2'))
REFERENCE_EVENTS = ONLINE_CACHE / "container_events.csv"
sys.path.insert(0, str(ONLINE_DIR))
from simulation import prepare_trace

PRE = {"Non-GPU": 85, "GPU-Enabled": 119}  # GPU Pre 119 (Train P90 119.0)
POST = {"Non-GPU": 146, "GPU-Enabled": 170}
NS_PER_MIN = 60_000_000_000
SCENARIOS = [
    ("원고 기준 / Baseline", 10, 15, 20, 0),
    ("CPU -2pp / CPU 8%", 8, 15, 20, 0),
    ("CPU +2pp / CPU 12%", 12, 15, 20, 0),
    ("메모리 -5pp / Memory lower", 10, 10, 15, 0),
    ("메모리 +5pp / Memory higher", 10, 20, 25, 0),
    ("GPU ≤1% / GPU tolerance", 10, 15, 20, 1),
]
GAP_SETTINGS = [1.0, 1.02, 2.0, 3.0, np.inf]  # Finite settings are retained diagnostics; infinity is current no-reset baseline.
for p in [USAGE_PATH, TRACE_PATH, REFERENCE_EVENTS]:
    if not p.is_file():
        raise FileNotFoundError(p)

def normalize_filename(value):
    name = str(value).replace("\\", "/").split("/")[-1].strip()
    for ext in (".csv", ".parquet"):
        if name.lower().endswith(ext):
            return name[:-len(ext)]
    return name

usage = pd.read_csv(USAGE_PATH, usecols=[
    "filename", "created_time", "terminated_time", "gpu_allocated(group)",
    "gpu_allocated(EA)", "cpu_allocated(Core)"], low_memory=False)
usage["key"] = usage.filename.map(normalize_filename)
if usage.key.isna().any() or usage.key.duplicated().any():
    raise ValueError("UsageLog key 결측 또는 중복")
for col in ["created_time", "terminated_time"]:
    usage[col] = pd.to_datetime(usage[col], errors="raise")
for col in ["gpu_allocated(EA)", "cpu_allocated(Core)"]:
    usage[col] = pd.to_numeric(usage[col], errors="raise")
    if usage[col].isna().any() or usage[col].lt(0).any():
        raise ValueError(f"할당량 결측·음수: {col}")
usage["population"] = "Excluded"
usage.loc[usage.created_time.ge("2023-03-01") & usage.created_time.lt("2025-12-01"), "population"] = "Train"
usage.loc[usage.created_time.ge("2026-03-01") & usage.created_time.lt("2026-09-01"), "population"] = "Test"
lookup = usage.set_index("key", verify_integrity=True)
parquet = pq.ParquetFile(TRACE_PATH)
if parquet.metadata.num_row_groups != _corrected_contract["files_post"] or parquet.metadata.num_rows != _corrected_contract["rows_post"]:
    raise ValueError("TimeSeries 크기가 기존 분석 입력과 다름")
group_rows = []
for group_id in range(parquet.metadata.num_row_groups):
    rg = parquet.metadata.row_group(group_id)
    name_stat, gpu_stat = rg.column(8).statistics, rg.column(6).statistics
    if name_stat is None or gpu_stat is None or name_stat.min != name_stat.max or gpu_stat.min != gpu_stat.max:
        raise ValueError(f"row group {group_id}: 단일 filename/GPU 그룹이 아님")
    filename = str(name_stat.min)
    key = normalize_filename(filename)
    if key not in lookup.index:
        raise ValueError(f"미연결 TimeSeries: {key}")
    meta = lookup.loc[key]
    if str(gpu_stat.min) != meta["gpu_allocated(group)"]:
        raise ValueError(f"GPU 그룹 불일치: {key}")
    if str(gpu_stat.min) not in {"0.0_GPU", "1.0_GPU", "2.0_GPU", "3.0_GPU", "4.0_GPU"}:
        raise ValueError(f"적격 외 GPU 그룹: {key}")
    group_rows.append((group_id, filename, key, meta.population,
                       str(gpu_stat.min), meta.terminated_time,
                       float(meta["gpu_allocated(EA)"]),
                       float(meta["cpu_allocated(Core)"]), rg.num_rows))
groups = pd.DataFrame(group_rows, columns=[
    "group_id", "filename", "key", "population", "gpu_group",
    "terminated_time", "gpu_allocated", "cpu_allocated", "sample_rows"])
groups["gpu_bin"] = np.where(groups.gpu_group.eq("0.0_GPU"), "Non-GPU", "GPU-Enabled")
if groups.key.duplicated().any() or groups.population.value_counts().to_dict() != LATEST_CONTRACT['population_counts']:
    raise ValueError("기존 population 불일치")
if int(groups.sample_rows.sum()) != _corrected_contract["rows_post"]:
    raise ValueError("row-group 행 합계 오류")
print("[RESULT] input_population_validation: PASS")
display(groups.groupby(["population", "gpu_bin"]).agg(
    containers=("key", "size"), samples=("sample_rows", "sum")
).rename_axis(index=["기간 / Population", "GPU 유형 / GPU type"])
 .rename(columns={"containers": "컨테이너 / Containers", "samples": "표본 / Samples"}))

# Cohort and applied IR2 timeouts (85/119/146/170) from the frozen cohort contract.
sys.path.insert(0, str(PROJECT_ROOT / "04_Analysis/revision"))
from cohort_contract import current_contract
LATEST_CONTRACT = current_contract()


In [ ]:
def first_online_event(trace, gpu_bin, max_gap_min=np.inf):
    """02의 online_decision과 같은 첫 관측 종료. 연속 간격만 인자로 분리."""
    if trace.empty:
        return pd.NaT, None, False
    times = trace.time.to_numpy(dtype="datetime64[ns]").astype(np.int64)
    states = trace.state.to_numpy()
    idle, active = states == "idle", states == "active"
    n = len(states)
    continuous = np.r_[False, np.diff(times) <= max_gap_min * NS_PER_MIN]
    previous_idle = np.r_[False, idle[:-1]]
    previous_active = np.r_[False, active[:-1]]
    new_run = idle & (~previous_idle | ~continuous)
    proposed = np.where(previous_active & continuous, np.arange(n)-1, np.arange(n))
    run_start = np.maximum.accumulate(np.where(new_run, proposed, -1))
    duration_ns = times - times[np.maximum(run_start, 0)]
    seen_active = np.maximum.accumulate(active)
    limit_ns = np.where(seen_active, POST[gpu_bin], PRE[gpu_bin]) * NS_PER_MIN
    hits = np.flatnonzero(idle & (duration_ns >= limit_ns))
    if not len(hits):
        return pd.NaT, None, False
    first = int(hits[0])
    kind = "Post" if seen_active[first] else "Pre"
    later_active = bool(active[first + 1:].any())
    return pd.Timestamp(trace.time.iloc[first]), kind, later_active

# 관측 경계 합성 사례: 정확히 기준에 도달한 idle만 종료하고,
# active는 timer를 끊는다. 현재 baseline은 관측 공백에 따른 reset이 없다; 유한 간격은 과거 보조 진단이다.
base = pd.Timestamp("2026-01-01")
def synthetic(states, minutes):
    return pd.DataFrame({"time": [base + pd.Timedelta(minutes=m) for m in minutes],
                         "state": states})
assert first_online_event(synthetic(["idle"]*(int(PRE["Non-GPU"])+1), list(range(int(PRE["Non-GPU"])+1))), "Non-GPU", 2)[1] == "Pre"
assert first_online_event(synthetic(["idle"]*int(PRE["Non-GPU"])+["active"], list(range(int(PRE["Non-GPU"])+1))), "Non-GPU", 2)[1] is None
assert first_online_event(synthetic(["idle","idle","idle"], [0,1,89]), "Non-GPU", 2)[1] is None
assert first_online_event(synthetic(["idle","idle","idle"], [0,1,89]), "Non-GPU", 100)[1] == "Pre"
assert first_online_event(synthetic(["idle","idle","idle"], [0,1,89]), "Non-GPU")[1] == "Pre"
assert first_online_event(synthetic(["active","idle","idle"], [0,1,180]), "Non-GPU")[1] == "Post"
print("[RESULT] synthetic_online_boundary_tests: PASS")


In [ ]:
scenario_rows, gap_rows, quality_rows, margin_rows = [], [], [], []
needed = ["time", "used_time(minute)", "cpu_util(%)", "mem_util(%)", "gpu_util(%)"]
for item in groups.itertuples(index=False):
    block = parquet.read_row_group(item.group_id, columns=needed).to_pandas()
    if len(block) != item.sample_rows or block[["time","used_time(minute)","cpu_util(%)","mem_util(%)"] + (["gpu_util(%)"] if item.gpu_bin == "GPU-Enabled" else [])].isna().any().any():
        raise ValueError(f"행 수·필수 결측: {item.key}")
    if block["used_time(minute)"].lt(0).any():
        raise ValueError(f"음수 경과시간: {item.key}")
    order = np.argsort(block["used_time(minute)"].to_numpy(float), kind="stable")
    elapsed = block["used_time(minute)"].to_numpy(float)[order]
    cpu = block["cpu_util(%)"].to_numpy(float)[order]
    memory = block["mem_util(%)"].to_numpy(float)[order]
    gpu = block["gpu_util(%)"].to_numpy(float)[order]
    assert item.gpu_bin == "Non-GPU" or (np.isfinite(gpu).all() and np.all(gpu >= 0)), "Integer-GPU utilization contract changed"
    end = float(elapsed[-1])
    pre_limit, post_limit = PRE[item.gpu_bin], POST[item.gpu_bin]
    sample_count = len(block)
    gpu_required = item.gpu_bin == "GPU-Enabled"
    base_mem = 20 if gpu_required else 15
    other_gpu = (gpu == 0) if gpu_required else np.ones(sample_count, dtype=bool)
    cpu_margin = int(((memory <= base_mem) & other_gpu & (cpu >= 8) & (cpu <= 12)).sum())
    memory_margin = int(((memory >= base_mem-5) & (memory <= base_mem+5)
                         & (cpu <= 10) & other_gpu).sum())
    gpu_margin = int(((gpu > 0) & (gpu <= 1) & (cpu <= 10)
                      & (memory <= 20)).sum()) if gpu_required else 0
    margin_rows.append((item.key, item.population, item.gpu_bin, sample_count,
                        cpu_margin, memory_margin, gpu_margin))
    for name, cpu_limit, mem_non, mem_gpu, gpu_limit in SCENARIOS:
        mem_limit = mem_gpu if gpu_required else mem_non
        idle = (cpu <= cpu_limit) & (memory <= mem_limit)
        if gpu_required:
            idle &= (gpu == 0) if gpu_limit == 0 else (gpu <= gpu_limit)
        active_indices = np.flatnonzero(~idle)
        normal = bool(len(active_indices))
        if normal:
            first_active = float(elapsed[active_indices[0]])
            last_active = float(elapsed[active_indices[-1]])
            pre_saved = max(first_active - 1 - pre_limit, 0)
            post_saved = max(end - last_active - post_limit, 0) if bool(idle[-1]) else 0
            pre_candidate = first_active - float(elapsed[0]) > pre_limit
        else:
            pre_saved = max(end - pre_limit, 0)
            post_saved, pre_candidate = 0, False
        saved_min = pre_saved + post_saved
        if saved_min > end + 1e-8:
            raise ValueError(f"절감시간이 분모 초과: {item.key}/{name}")
        scenario_rows.append((item.key, item.population, item.gpu_bin, name,
                              normal, bool(pre_candidate), pre_saved, post_saved,
                              item.gpu_allocated * end, item.cpu_allocated * end,
                              item.gpu_allocated * saved_min, item.cpu_allocated * saved_min))
    block["filename"] = item.filename
    block["gpu_allocated(group)"] = item.gpu_group
    prepared, audit = prepare_trace(block, item.gpu_bin, item.terminated_time)
    times = prepared.time.to_numpy(dtype="datetime64[ns]").astype(np.int64)
    intervals = np.diff(times)
    if (intervals <= 0).any():
        raise ValueError(f"정리 후 시각 중복·역전: {item.key}")
    quality_rows.append((item.key, item.population, item.gpu_bin, len(prepared),
                         len(intervals), int((intervals > 1*NS_PER_MIN).sum()),
                         int(((intervals > NS_PER_MIN) & (intervals <= 1.02*NS_PER_MIN)).sum()),
                         int((intervals > 1.02*NS_PER_MIN).sum()),
                         int((intervals > 2*NS_PER_MIN).sum()),
                         int((intervals > 3*NS_PER_MIN).sum()),
                         float(intervals.max()/NS_PER_MIN) if len(intervals) else 0,
                         int(audit["duplicate_rows"]), int(audit["conflicting_timestamps"]),
                         int(audit["rows_after_termination"]),
                         int(prepared.state.eq("unknown").sum())))
    for max_gap in GAP_SETTINGS:
        event_time, kind, later_active = first_online_event(prepared, item.gpu_bin, max_gap)
        gap_rows.append((item.key, item.population, item.gpu_bin, max_gap,
                         event_time, kind, later_active))

scenario_detail = pd.DataFrame(scenario_rows, columns=[
    "key", "population", "gpu_bin", "scenario", "normal", "pre_candidate",
    "pre_saved_min", "post_saved_min", "gpu_baseline_min", "cpu_baseline_min",
    "gpu_saved_min", "cpu_saved_min"])
gap_detail = pd.DataFrame(gap_rows, columns=[
    "key", "population", "gpu_bin", "max_gap_min", "online_time",
    "online_kind", "later_active"])
quality_detail = pd.DataFrame(quality_rows, columns=[
    "key", "population", "gpu_bin", "samples_after_cleanup", "intervals",
    "gap_over_1", "gap_gt_1_le_1p02", "gap_over_1p02",
    "gap_over_2", "gap_over_3", "max_gap_min",
    "duplicate_rows", "conflicting_timestamps", "rows_after_termination",
    "unknown_samples"])
margin_detail = pd.DataFrame(margin_rows, columns=[
    "key", "population", "gpu_bin", "samples", "cpu_near", "memory_near", "gpu_near"])
if len(scenario_detail) != len(groups)*len(SCENARIOS) or len(gap_detail) != len(groups)*len(GAP_SETTINGS):
    raise ValueError("시나리오 상세행 수 오류")
print("[RESULT] full_trace_scan: PASS")
print("Containers:", len(groups), "Samples:", int(groups.sample_rows.sum()),
      "Scenario rows:", len(scenario_detail), "Online rows:", len(gap_detail))


In [ ]:
_corrected_contract=__import__('json').loads((ROOT/'inputs/preprocessing_validation.json').read_text())
quality = quality_detail.groupby(["population", "gpu_bin"]).agg(
    containers=("key", "size"), intervals=("intervals", "sum"),
    gaps_gt_1=("gap_over_1", "sum"),
    jitter_band_1_to_1p02=("gap_gt_1_le_1p02", "sum"),
    gaps_gt_1p02=("gap_over_1p02", "sum"),
    gaps_gt_2=("gap_over_2", "sum"),
    gaps_gt_3=("gap_over_3", "sum"), max_gap_min=("max_gap_min", "max"),
    containers_with_gap_gt_2=("gap_over_2", lambda s: int(s.gt(0).sum())),
    duplicate_rows=("duplicate_rows", "sum"),
    conflicting_timestamps=("conflicting_timestamps", "sum"),
    rows_after_termination=("rows_after_termination", "sum"),
    unknown_samples=("unknown_samples", "sum"))
quality["gaps_gt_2_pct"] = quality.gaps_gt_2 / quality.intervals * 100
margins = margin_detail.groupby(["population", "gpu_bin"])[
    ["samples", "cpu_near", "memory_near", "gpu_near"]].sum()
for col in ["cpu_near", "memory_near", "gpu_near"]:
    margins[col + "_pct"] = margins[col] / margins.samples * 100
if int(quality.containers.sum()) != LATEST_CONTRACT['row_groups'] or int(margins.samples.sum()) != _corrected_contract["rows_post"]:
    raise ValueError("샘플링·근접 표본 분모 합계 오류")
if not (quality.gaps_gt_1 == quality.jitter_band_1_to_1p02 + quality.gaps_gt_1p02).all():
    raise ValueError("1분 초과 간격 구간 분해 오류")
print("[RESULT] sampling_quality_validation: PASS")
quality_labels = {"containers":"컨테이너 / Containers", "intervals":"인접 간격 / Intervals",
    "gaps_gt_1":">1분 / >1 min", "jitter_band_1_to_1p02":"1~1.02분 / Jitter band",
    "gaps_gt_1p02":">1.02분 / >1.02 min", "gaps_gt_2":">2분 / >2 min",
    "gaps_gt_3":">3분 / >3 min", "max_gap_min":"최장 공백 / Max gap (min)",
    "containers_with_gap_gt_2":">2분 경험 컨테이너 / Containers",
    "duplicate_rows":"중복 행 / Duplicates", "conflicting_timestamps":"시각 충돌 / Conflicts",
    "rows_after_termination":"종료 후 행 / Post-termination",
    "unknown_samples":"미확인 표본 / Unknown", "gaps_gt_2_pct":">2분 비율 / Gap rate (%)"}
display(quality.rename_axis(index=["기간 / Population", "GPU 유형 / GPU type"])
        .rename(columns=quality_labels).round(2))
print("[RESULT] cutoff_margin_profile: PASS")
margin_labels = {"samples":"표본 / Samples", "cpu_near":"CPU 경계 인접 / Near CPU",
    "memory_near":"메모리 경계 인접 / Near memory", "gpu_near":"GPU 경계 인접 / Near GPU",
    "cpu_near_pct":"CPU 인접 비율 / CPU (%)",
    "memory_near_pct":"메모리 인접 비율 / Memory (%)",
    "gpu_near_pct":"GPU 인접 비율 / GPU (%)"}
display(margins.rename_axis(index=["기간 / Population", "GPU 유형 / GPU type"])
        .rename(columns=margin_labels).round(3))


### 결과
## 2. 이용률 cutoff 한 요인 민감도

In [ ]:
scenario_summary = scenario_detail.groupby(["scenario", "population"]).agg(
    containers=("key", "size"), normal=("normal", "sum"),
    pre_candidates=("pre_candidate", "sum"),
    gpu_den=("gpu_baseline_min", "sum"), cpu_den=("cpu_baseline_min", "sum"),
    gpu_saved=("gpu_saved_min", "sum"), cpu_saved=("cpu_saved_min", "sum"))
scenario_summary["low_util"] = scenario_summary.containers - scenario_summary.normal
scenario_summary["candidate_pct_normal"] = scenario_summary.pre_candidates / scenario_summary.normal * 100
scenario_summary["gpu_saved_pct"] = scenario_summary.gpu_saved / scenario_summary.gpu_den * 100
scenario_summary["cpu_saved_pct"] = scenario_summary.cpu_saved / scenario_summary.cpu_den * 100
scenario_summary = scenario_summary.reindex(pd.MultiIndex.from_product(
    [[s[0] for s in SCENARIOS], ["Train", "Test"]], names=["scenario", "population"]))
base = scenario_summary.loc["원고 기준 / Baseline"]
for col in ["gpu_den", "cpu_den"]:
    if not scenario_summary[col].groupby("population").nunique().eq(1).all():
        raise ValueError(f"시나리오별 분모가 달라짐: {col}")
expect = {}
for pop in ['Train','Test']:
    a=LATEST_CONTRACT['class_counts'][pop]; r=LATEST_CONTRACT['resources'][pop]
    expect[pop]={'containers':a['total'],'normal':a['normal'],'low_util':a['low_util'],
        'pre_candidates':LATEST_CONTRACT['pre_candidates'][pop],
        'gpu_den':r['GPU']['baseline'],'cpu_den':r['CPU']['baseline'],
        'gpu_saved':r['GPU']['total_saved'],'cpu_saved':r['CPU']['total_saved']}
for pop, checks in expect.items():
    for col, wanted in checks.items():
        actual = float(base.loc[pop,col])
        if not np.isclose(actual, wanted, rtol=0, atol=.11):
            raise ValueError(f"01 baseline 재현 실패: {pop}/{col}: {actual} vs {wanted}")
for col in ["gpu_saved_pct", "cpu_saved_pct", "candidate_pct_normal"]:
    scenario_summary[col+"_delta"] = scenario_summary[col] - scenario_summary.index.get_level_values(
        "population").map(base[col])
print("[RESULT] cutoff_sensitivity_baseline_reconciliation: PASS")
scenario_labels = {"containers":"컨테이너 / Containers","low_util":"저이용 / Low-util",
    "normal":"정상 / Normal","pre_candidates":"Pre 후보 / Pre candidates",
    "candidate_pct_normal":"Normal 중 후보 / Candidate (%)",
    "gpu_saved_pct":"GPU 절감률 / GPU saved (%)",
    "gpu_saved_pct_delta":"GPU 변화 / GPU delta (pp)",
    "cpu_saved_pct":"CPU 절감률 / CPU saved (%)",
    "cpu_saved_pct_delta":"CPU 변화 / CPU delta (pp)"}
display(scenario_summary[["containers","low_util","normal","pre_candidates",
    "candidate_pct_normal","gpu_saved_pct","gpu_saved_pct_delta"]].rename_axis(
    index=["시나리오 / Scenario","기간 / Population"]).rename(columns=scenario_labels).round(3))


### 결과
## 3. 온라인 연속 관측 간격 1.00/1.02/2.00/3.00분 민감도

In [ ]:
reference = pd.read_csv(REFERENCE_EVENTS, usecols=[
    "filename", "population", "online_time", "online_kind", "potential_interruption"],
    low_memory=False)
reference["online_time"] = pd.to_datetime(reference.online_time)
baseline_online = gap_detail.loc[np.isinf(gap_detail.max_gap_min)].copy()
compared = baseline_online.merge(reference, left_on="key", right_on="filename",
                                  how="outer", validate="one_to_one", indicator=True,
                                  suffixes=("_new", "_reference"))
if not compared._merge.eq("both").all() or len(compared) != LATEST_CONTRACT["row_groups"]:
    raise ValueError("02 온라인 baseline 컨테이너 연결 실패")
time_match = (compared.online_time_new.fillna(pd.Timestamp("1900-01-01"))
              == compared.online_time_reference.fillna(pd.Timestamp("1900-01-01")))
kind_match = compared.online_kind_new.fillna("None").eq(compared.online_kind_reference.fillna("None"))
activity_match = compared.later_active.eq(compared.potential_interruption)
if not (time_match & kind_match & activity_match).all():
    raise ValueError(f"02 온라인 컨테이너별 불일치: time={int((~time_match).sum())}, "
                     f"kind={int((~kind_match).sum())}, activity={int((~activity_match).sum())}")
online_summary = gap_detail.groupby(["max_gap_min","population","gpu_bin"]).agg(
    containers=("key","size"),
    terminated=("online_time",lambda s:int(s.notna().sum())),
    later_active=("later_active","sum"))
online_summary["later_active_pct_of_terminated"] = (
    online_summary.later_active / online_summary.terminated.replace(0,np.nan) * 100)
online_all = gap_detail.groupby(["max_gap_min","population"]).agg(
    containers=("key","size"),
    terminated=("online_time",lambda s:int(s.notna().sum())),
    later_active=("later_active","sum"))
online_all["later_active_pct_of_terminated"] = (
    online_all.later_active / online_all.terminated.replace(0,np.nan) * 100)
for pop, (term, later) in {pop:(v["terminated"],v["returns"]) for pop,v in LATEST_CONTRACT["online_counts"].items()}.items():
    row = online_all.loc[(np.inf,pop)]
    if (int(row.terminated),int(row.later_active)) != (term,later):
        raise ValueError(f"02 집계 불일치: {pop}")
if not online_summary.groupby(["max_gap_min","population"])[["containers","terminated","later_active"]].sum().eq(
        online_all[["containers","terminated","later_active"]]).all().all():
    raise ValueError("GPU 유형 합계 불일치")
print("[RESULT] online_gap_baseline_reconciliation: PASS")
online_labels = {"containers":"컨테이너 / Containers","terminated":"첫 종료 / First termination",
    "later_active":"이후 활동 / Later activity",
    "later_active_pct_of_terminated":"종료 중 이후 활동 / Later active (%)"}
display(online_all.rename_axis(index=["최대 연속 간격 / Max gap (min)", "기간 / Population"])
        .rename(columns=online_labels).round(2))
print("[RESULT] online_gap_gpu_subgroups: PASS")
display(online_summary.rename_axis(index=["최대 연속 간격 / Max gap (min)",
                                           "기간 / Population", "GPU 유형 / GPU type"])
        .rename(columns=online_labels).round(2))


### 결과

In [ ]:
# Deterministic exports of existing retained summary tables (no new analysis).
quality.to_csv(RESULTS / "10_input_quality.csv", index=True)
scenario_summary.to_csv(RESULTS / "10_utilization_cutoff_sensitivity.csv", index=True)
online_summary.to_csv(RESULTS / "10_sampling_continuity_summary.csv", index=True)
online_all.to_csv(RESULTS / "10_sampling_continuity_all.csv", index=True)
